# 07 - DrugCentral EGFR Indications

Goal: enrich EGFR therapy candidates with DrugCentral activity and indication context.

Flow: EGFR -> DrugCentral activity rows -> structure names -> OMOP indications -> cleaned CSVs.

Outputs: `egfr_drugcentral_activity.csv`, `egfr_drugcentral_indications.csv`, `egfr_drugcentral_summary.csv`

### 1. Test notebook environment

In [25]:
import json
import re
import sys
import time
from pathlib import Path

import pandas as pd
import requests

print("Notebook is working")
print("Python executable:", sys.executable)

Notebook is working
Python executable: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/.venv/bin/python


### 2. Set project folders

In [26]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "drugcentral"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Processed data folder:", PROCESSED_DIR)

Project root: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant
Raw data folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/drugcentral
Processed data folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed


### 3. Load existing EGFR drug recommendations

In [27]:
target_name = "EGFR"
recommendations_file = PROCESSED_DIR / "egfr_drug_recommendations.csv"

if not recommendations_file.exists():
    raise FileNotFoundError(f"Missing required input: {recommendations_file}")

drug_recommendations_df = pd.read_csv(recommendations_file)
project_drug_names = sorted(drug_recommendations_df["drug_name"].dropna().astype(str).unique())

print("Project drugs:", len(project_drug_names))
drug_recommendations_df.head()

Project drugs: 76


,drug_name,molecule_chembl_id,action_type,mechanism_of_action,approval_status,max_phase,target_name
0,PANITUMUMAB,CHEMBL1201827,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor
1,CETUXIMAB,CHEMBL1201577,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor
2,ERLOTINIB HYDROCHLORIDE,CHEMBL1079742,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor
3,GEFITINIB,CHEMBL939,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor
4,LAPATINIB DITOSYLATE,CHEMBL1201179,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor


### 4. Helper functions

In [30]:
DRUGCENTRAL_API_BASE = "https://uxn2ycvimg.us-east-2.awsapprunner.com"


def normalize_name(value):
    """Normalize drug names for safer matching across datasets."""
    if pd.isna(value):
        return ""
    return re.sub(r"[^A-Z0-9]+", "", str(value).upper())


def get_json(path, retries=3, pause=1):
    """GET JSON from DrugCentral API path with simple retries."""
    url = f"{DRUGCENTRAL_API_BASE}{path}"
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=(10, 60))
            if response.status_code == 200:
                return response.json()
            print(f"  {path} attempt {attempt + 1}: HTTP {response.status_code}, retrying...")
        except requests.exceptions.RequestException as error:
            print(f"  {path} attempt {attempt + 1}: {type(error).__name__}, retrying...")
        time.sleep(pause)
    return None


project_drug_norms = {normalize_name(name) for name in project_drug_names}
print("Normalized project drug names:", len(project_drug_norms))

Normalized project drug names: 76


### 5. Fetch DrugCentral EGFR activity and target records

In [31]:
activity_rows = get_json(f"/act_table_full/gene/{target_name}")
target_component_rows = get_json(f"/target_component/gene/{target_name}")

if activity_rows is None:
    raise RuntimeError("DrugCentral activity request failed.")
if target_component_rows is None:
    raise RuntimeError("DrugCentral target component request failed.")

with (RAW_DIR / "egfr_drugcentral_activity_raw.json").open("w") as f:
    json.dump(activity_rows, f, indent=2)

with (RAW_DIR / "egfr_drugcentral_target_component_raw.json").open("w") as f:
    json.dump(target_component_rows, f, indent=2)

activity_df = pd.DataFrame(activity_rows)
target_component_df = pd.DataFrame(target_component_rows)

print("Activity rows:", len(activity_df))
print("Target component rows:", len(target_component_df))
activity_df.head()

Activity rows: 75
Target component rows: 2


,gene,moa,act_ref_id,accession,swissprot,moa_source,target_name,act_value,act_source_url,target_id,...,action_type,first_in_class,act_id,act_comment,moa_ref_id,target_class,act_source,tdl,organism,relation
0,EGFR,NaN,NaN,P00533,EGFR_HUMAN,NaN,Epidermal growth factor receptor,5.269,NaN,387,...,NaN,NaN,33909,"DRUGMATRIX: Protein Tyrosine Kinase, EGF Recep...",NaN,Kinase,DRUG MATRIX,Tclin,Homo sapiens,=
1,EGFR,NaN,NaN,P00533,EGFR_HUMAN,NaN,Epidermal growth factor receptor,5.218,NaN,387,...,NaN,NaN,33874,"DRUGMATRIX: Protein Tyrosine Kinase, EGF Recep...",NaN,Kinase,DRUG MATRIX,Tclin,Homo sapiens,=
2,EGFR,NaN,NaN,P00533,EGFR_HUMAN,NaN,Epidermal growth factor receptor,5.480,NaN,387,...,NaN,NaN,210376,"DRUGMATRIX: Protein Tyrosine Kinase, EGF Recep...",NaN,Kinase,CHEMBL,Tclin,Homo sapiens,=
3,EGFR,NaN,NaN,P00533,EGFR_HUMAN,NaN,Epidermal growth factor receptor,4.540,NaN,387,...,NaN,NaN,34222,"DRUGMATRIX: Protein Tyrosine Kinase, EGF Recep...",NaN,Kinase,DRUG MATRIX,Tclin,Homo sapiens,=
4,EGFR,NaN,NaN,P00533,EGFR_HUMAN,NaN,Epidermal growth factor receptor,5.979,NaN,387,...,NaN,NaN,34437,"DRUGMATRIX: Protein Tyrosine Kinase, EGF Recep...",NaN,Kinase,DRUG MATRIX,Tclin,Homo sapiens,=


### 6. Fetch structure names and OMOP relationships for EGFR activity compounds

In [32]:
struct_ids = sorted({int(value) for value in activity_df["struct_id"].dropna().unique()})
print("Unique DrugCentral structure IDs:", len(struct_ids))

structure_rows = []
relationship_rows = []

for index, struct_id in enumerate(struct_ids, start=1):
    structure_response = get_json(f"/structures/id/{struct_id}") or []
    relationship_response = get_json(f"/omop_relationship/struct_id/{struct_id}") or []

    for row in structure_response:
        row["struct_id"] = struct_id
        structure_rows.append(row)

    for row in relationship_response:
        row["struct_id"] = struct_id
        relationship_rows.append(row)

    if index % 20 == 0:
        print(f"Fetched {index}/{len(struct_ids)} structure IDs")
    time.sleep(0.05)

with (RAW_DIR / "egfr_drugcentral_structures_raw.json").open("w") as f:
    json.dump(structure_rows, f, indent=2)

with (RAW_DIR / "egfr_drugcentral_relationships_raw.json").open("w") as f:
    json.dump(relationship_rows, f, indent=2)

structures_df = pd.DataFrame(structure_rows)
relationships_df = pd.DataFrame(relationship_rows)

print("Structure rows:", len(structures_df))
print("Relationship rows:", len(relationships_df))
structures_df.head()

Unique DrugCentral structure IDs: 74
Fetched 20/74 structure IDs
  /omop_relationship/struct_id/2106 attempt 1: HTTP 404, retrying...
  /omop_relationship/struct_id/2106 attempt 2: HTTP 404, retrying...
  /omop_relationship/struct_id/2106 attempt 3: HTTP 404, retrying...
  /omop_relationship/struct_id/2311 attempt 1: HTTP 404, retrying...
  /omop_relationship/struct_id/2311 attempt 2: HTTP 404, retrying...
  /omop_relationship/struct_id/2311 attempt 3: HTTP 404, retrying...
  /omop_relationship/struct_id/2536 attempt 1: HTTP 404, retrying...
  /omop_relationship/struct_id/2536 attempt 2: HTTP 404, retrying...
  /omop_relationship/struct_id/2536 attempt 3: HTTP 404, retrying...
  /omop_relationship/struct_id/3032 attempt 1: HTTP 404, retrying...
  /omop_relationship/struct_id/3032 attempt 2: HTTP 404, retrying...
  /omop_relationship/struct_id/3032 attempt 3: HTTP 404, retrying...
  /omop_relationship/struct_id/3533 attempt 1: HTTP 404, retrying...
  /omop_relationship/struct_id/3533 at

,clogp,rotb,status,alogs,mrdef,o_n,cd_formula,cas_reg_no,arom_c,oh_nh,...,rgb,cd_molweight,no_formulations,halogen,fda_labels,stem,molfile,hetero_sp2_c,inchikey,struct_id
0,5.84,8.0,NaN,-5.58,Antihistamine drug now withdrawn from the mark...,5.0,C28H31FN4O,68844-77-9,19.0,1.0,...,26.0,458.581,NaN,1.0,NaN,-astine,\n -INDIGO-08151712112D\n\n 34 38 0 0 0 0...,0.0,GXDALQBWZGODGZ-UHFFFAOYSA-N,249
1,NaN,NaN,NaN,NaN,A complex of cyclic peptide antibiotics produc...,NaN,C261H406N68O64S4,1405-87-4,NaN,NaN,...,NaN,5648.770,653.0,NaN,623.0,NaN,NaN,NaN,NaN,281
2,7.08,13.0,OFP,-5.48,a nonpeptide angiotensin II receptor antagonist,12.0,C33H34N6O6,145040-37-5,20.0,1.0,...,36.0,610.671,77.0,0.0,27.0,-sartan,\n -INDIGO-08151712112D\n\n 45 50 0 0 0 0...,2.0,GHOSNRCGJFBJIB-UHFFFAOYSA-N,475
3,5.50,4.0,OFP,-4.88,The prototypical phenothiazine antipsychotic d...,2.0,C17H19ClN2S,50-53-3,12.0,0.0,...,16.0,318.860,77.0,1.0,30.0,NaN,\n -INDIGO-08151712102D\n\n 21 23 0 0 0 0...,0.0,ZPEIMTDSQAKGNT-UHFFFAOYSA-N,621
4,7.15,9.0,OFP,-5.99,A triphenyl ethylene stilbene derivative which...,2.0,C26H28ClNO,911-45-5,18.0,0.0,...,19.0,405.970,8.0,1.0,8.0,-mifene,\n -INDIGO-08151712112D\n\n 29 31 0 0 0 0...,0.0,GKIRPKYJQBWNGO-UHFFFAOYSA-N,700


### 7. Build processed DrugCentral activity table

In [33]:
structure_name_df = structures_df[["id", "name", "cas_reg_no", "fda_labels", "no_formulations"]].drop_duplicates().rename(
    columns={"id": "struct_id", "name": "drugcentral_name"}
)

drugcentral_activity_df = activity_df.merge(structure_name_df, on="struct_id", how="left")
if "target_name" in drugcentral_activity_df.columns:
    drugcentral_activity_df = drugcentral_activity_df.rename(columns={"target_name": "target_full_name"})
drugcentral_activity_df["target_name"] = target_name
drugcentral_activity_df["normalised_drug_name"] = drugcentral_activity_df["drugcentral_name"].apply(normalize_name)
drugcentral_activity_df["matches_project_drug"] = drugcentral_activity_df["normalised_drug_name"].isin(project_drug_norms)
drugcentral_activity_df["source"] = "DrugCentral"

activity_columns = [
    "target_name", "gene", "accession", "swissprot", "target_full_name", "target_class", "tdl",
    "struct_id", "drugcentral_name", "normalised_drug_name", "matches_project_drug",
    "act_type", "relation", "act_value", "act_unit", "act_source", "act_comment",
    "action_type", "moa", "cas_reg_no", "fda_labels", "no_formulations", "source",
]
activity_columns = [column for column in activity_columns if column in drugcentral_activity_df.columns]
drugcentral_activity_df = drugcentral_activity_df[activity_columns]

activity_file = PROCESSED_DIR / "egfr_drugcentral_activity.csv"
drugcentral_activity_df.to_csv(activity_file, index=False)

print("Saved:", activity_file)
print("Rows:", len(drugcentral_activity_df))
drugcentral_activity_df.head()

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_drugcentral_activity.csv
Rows: 75


,target_name,gene,accession,swissprot,target_full_name,target_class,tdl,struct_id,drugcentral_name,normalised_drug_name,...,act_value,act_unit,act_source,act_comment,action_type,moa,cas_reg_no,fda_labels,no_formulations,source
0,EGFR,EGFR,P00533,EGFR_HUMAN,Epidermal growth factor receptor,Kinase,Tclin,281,bacitracin,BACITRACIN,...,5.269,None,DRUG MATRIX,"DRUGMATRIX: Protein Tyrosine Kinase, EGF Recep...",NaN,NaN,1405-87-4,623.0,653.0,DrugCentral
1,EGFR,EGFR,P00533,EGFR_HUMAN,Epidermal growth factor receptor,Kinase,Tclin,249,astemizole,ASTEMIZOLE,...,5.218,None,DRUG MATRIX,"DRUGMATRIX: Protein Tyrosine Kinase, EGF Recep...",NaN,NaN,68844-77-9,NaN,NaN,DrugCentral
2,EGFR,EGFR,P00533,EGFR_HUMAN,Epidermal growth factor receptor,Kinase,Tclin,475,candesartan cilexetil,CANDESARTANCILEXETIL,...,5.480,None,CHEMBL,"DRUGMATRIX: Protein Tyrosine Kinase, EGF Recep...",NaN,NaN,145040-37-5,27.0,77.0,DrugCentral
3,EGFR,EGFR,P00533,EGFR_HUMAN,Epidermal growth factor receptor,Kinase,Tclin,621,chlorpromazine,CHLORPROMAZINE,...,4.540,None,DRUG MATRIX,"DRUGMATRIX: Protein Tyrosine Kinase, EGF Recep...",NaN,NaN,50-53-3,30.0,77.0,DrugCentral
4,EGFR,EGFR,P00533,EGFR_HUMAN,Epidermal growth factor receptor,Kinase,Tclin,700,clomifene,CLOMIFENE,...,5.979,None,DRUG MATRIX,"DRUGMATRIX: Protein Tyrosine Kinase, EGF Recep...",NaN,NaN,911-45-5,8.0,8.0,DrugCentral


### 8. Build processed DrugCentral indication table

In [34]:
if relationships_df.empty:
    drugcentral_indications_df = pd.DataFrame(columns=[
        "target_name", "struct_id", "drugcentral_name", "relationship_name", "concept_name",
        "snomed_full_name", "snomed_conceptid", "umls_cui", "source",
    ])
else:
    relationship_name_df = relationships_df.merge(structure_name_df, on="struct_id", how="left")
    relationship_name_df["target_name"] = target_name
    relationship_name_df["source"] = "DrugCentral"
    drugcentral_indications_df = relationship_name_df[[
        "target_name", "struct_id", "drugcentral_name", "relationship_name", "concept_name",
        "snomed_full_name", "snomed_conceptid", "umls_cui", "source",
    ]].copy()

indications_file = PROCESSED_DIR / "egfr_drugcentral_indications.csv"
drugcentral_indications_df.to_csv(indications_file, index=False)

print("Saved:", indications_file)
print("Rows:", len(drugcentral_indications_df))
drugcentral_indications_df.head()

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_drugcentral_indications.csv
Rows: 821


,target_name,struct_id,drugcentral_name,relationship_name,concept_name,snomed_full_name,snomed_conceptid,umls_cui,source
0,EGFR,249,astemizole,indication,Sneezing,Sneezing,76067001.0,C0037383,DrugCentral
1,EGFR,249,astemizole,indication,Allergic conjunctivitis,Allergic conjunctivitis,473460002.0,C0009766,DrugCentral
2,EGFR,249,astemizole,indication,Chronic idiopathic urticaria,Chronic idiopathic urticaria,302162004.0,C0578870,DrugCentral
3,EGFR,249,astemizole,indication,Allergic rhinitis,Allergic rhinitis,61582004.0,C2607914,DrugCentral
4,EGFR,249,astemizole,off-label use,Urticaria,Urticaria,126485001.0,C0042109,DrugCentral


### 9. Build per-drug DrugCentral summary

In [35]:
def safe_join(values, limit=5):
    cleaned = [str(value) for value in values if pd.notna(value) and str(value).strip()]
    unique_values = list(dict.fromkeys(cleaned))
    return " | ".join(unique_values[:limit])


relationship_summary_df = drugcentral_indications_df.groupby(["struct_id", "drugcentral_name"], dropna=False).agg(
    indication_count=("relationship_name", lambda s: int((s == "indication").sum())),
    off_label_count=("relationship_name", lambda s: int((s == "off-label use").sum())),
    contraindication_count=("relationship_name", lambda s: int((s == "contraindication").sum())),
    top_indications=("concept_name", safe_join),
).reset_index() if not drugcentral_indications_df.empty else pd.DataFrame(columns=[
    "struct_id", "drugcentral_name", "indication_count", "off_label_count", "contraindication_count", "top_indications"
])

activity_summary_df = drugcentral_activity_df.groupby(["struct_id", "drugcentral_name"], dropna=False).agg(
    activity_count=("struct_id", "size"),
    min_activity_value=("act_value", "min"),
    activity_types=("act_type", safe_join),
    activity_sources=("act_source", safe_join),
    matches_project_drug=("matches_project_drug", "max"),
).reset_index()

drugcentral_summary_df = activity_summary_df.merge(
    relationship_summary_df,
    on=["struct_id", "drugcentral_name"],
    how="left",
)

for column in ["indication_count", "off_label_count", "contraindication_count"]:
    drugcentral_summary_df[column] = drugcentral_summary_df[column].fillna(0).astype(int)
drugcentral_summary_df["top_indications"] = drugcentral_summary_df["top_indications"].fillna("")
drugcentral_summary_df["target_name"] = target_name
drugcentral_summary_df["has_drugcentral_indication"] = drugcentral_summary_df["indication_count"] > 0
drugcentral_summary_df["drugcentral_evidence_score"] = (
    0.4 * drugcentral_summary_df["matches_project_drug"].astype(float)
    + 0.3 * (drugcentral_summary_df["activity_count"] > 0).astype(float)
    + 0.3 * drugcentral_summary_df["has_drugcentral_indication"].astype(float)
)

summary_columns = [
    "target_name", "struct_id", "drugcentral_name", "matches_project_drug", "activity_count",
    "min_activity_value", "activity_types", "activity_sources", "indication_count", "off_label_count",
    "contraindication_count", "top_indications", "has_drugcentral_indication", "drugcentral_evidence_score",
]
drugcentral_summary_df = drugcentral_summary_df[summary_columns].sort_values(
    ["matches_project_drug", "drugcentral_evidence_score", "activity_count"], ascending=False
).reset_index(drop=True)

summary_file = PROCESSED_DIR / "egfr_drugcentral_summary.csv"
drugcentral_summary_df.to_csv(summary_file, index=False)

print("Saved:", summary_file)
print("Rows:", len(drugcentral_summary_df))
drugcentral_summary_df.head(20)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_drugcentral_summary.csv
Rows: 74


,target_name,struct_id,drugcentral_name,matches_project_drug,activity_count,min_activity_value,activity_types,activity_sources,indication_count,off_label_count,contraindication_count,top_indications,has_drugcentral_indication,drugcentral_evidence_score
0,EGFR,5062,osimertinib,True,2,7.090,IC50,CHEMBL | SCIENTIFIC LITERATURE,1,0,0,"Non-small cell lung cancer, positive for epide...",True,1.0
1,EGFR,1282,gefitinib,True,1,9.400,Ki,CHEMBL,2,0,7,Fibrosis of lung | Breastfeeding (mother) | Ac...,True,1.0
2,EGFR,4954,cetuximab,True,1,9.410,Kd,IUPHAR,7,0,0,Squamous cell carcinoma of mouth | Squamous ce...,True,1.0
3,EGFR,4955,panitumumab,True,1,10.300,Kd,IUPHAR,1,0,0,Secondary malignant neoplasm of colon,True,1.0
4,EGFR,5071,necitumumab,True,1,9.490,Kd,IUPHAR,2,0,0,Non-small cell lung cancer | Squamous non-smal...,True,1.0
5,EGFR,5209,icotinib,True,1,8.699,IC50,SCIENTIFIC LITERATURE,1,0,0,Non-small cell lung cancer,True,1.0
6,EGFR,5210,olmutinib,True,1,8.036,IC50,SCIENTIFIC LITERATURE,1,0,0,Non-small cell lung cancer,True,1.0
7,EGFR,5233,brigatinib,True,1,7.174,IC50,SCIENTIFIC LITERATURE,2,0,0,Non-small cell lung cancer | Advanced/recurren...,True,1.0
8,EGFR,5252,neratinib,True,1,7.036,IC50,SCIENTIFIC LITERATURE,1,0,0,HER2-positive carcinoma of breast,True,1.0
9,EGFR,5297,dacomitinib,True,1,8.222,IC50,SCIENTIFIC LITERATURE,2,0,0,"Non-small cell lung cancer, positive for epide...",True,1.0


### 10. Final result

In [36]:
print("DrugCentral EGFR Enrichment Complete")
print("=" * 70)
print("Target:", target_name)
print("Activity rows:", len(drugcentral_activity_df))
print("Indication/relationship rows:", len(drugcentral_indications_df))
print("Summary rows:", len(drugcentral_summary_df))
print("Processed files:")
print("-", activity_file)
print("-", indications_file)
print("-", summary_file)
display(drugcentral_summary_df.head(20))

DrugCentral EGFR Enrichment Complete
Target: EGFR
Activity rows: 75
Indication/relationship rows: 821
Summary rows: 74
Processed files:
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_drugcentral_activity.csv
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_drugcentral_indications.csv
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_drugcentral_summary.csv


,target_name,struct_id,drugcentral_name,matches_project_drug,activity_count,min_activity_value,activity_types,activity_sources,indication_count,off_label_count,contraindication_count,top_indications,has_drugcentral_indication,drugcentral_evidence_score
0,EGFR,5062,osimertinib,True,2,7.090,IC50,CHEMBL | SCIENTIFIC LITERATURE,1,0,0,"Non-small cell lung cancer, positive for epide...",True,1.0
1,EGFR,1282,gefitinib,True,1,9.400,Ki,CHEMBL,2,0,7,Fibrosis of lung | Breastfeeding (mother) | Ac...,True,1.0
2,EGFR,4954,cetuximab,True,1,9.410,Kd,IUPHAR,7,0,0,Squamous cell carcinoma of mouth | Squamous ce...,True,1.0
3,EGFR,4955,panitumumab,True,1,10.300,Kd,IUPHAR,1,0,0,Secondary malignant neoplasm of colon,True,1.0
4,EGFR,5071,necitumumab,True,1,9.490,Kd,IUPHAR,2,0,0,Non-small cell lung cancer | Squamous non-smal...,True,1.0
5,EGFR,5209,icotinib,True,1,8.699,IC50,SCIENTIFIC LITERATURE,1,0,0,Non-small cell lung cancer,True,1.0
6,EGFR,5210,olmutinib,True,1,8.036,IC50,SCIENTIFIC LITERATURE,1,0,0,Non-small cell lung cancer,True,1.0
7,EGFR,5233,brigatinib,True,1,7.174,IC50,SCIENTIFIC LITERATURE,2,0,0,Non-small cell lung cancer | Advanced/recurren...,True,1.0
8,EGFR,5252,neratinib,True,1,7.036,IC50,SCIENTIFIC LITERATURE,1,0,0,HER2-positive carcinoma of breast,True,1.0
9,EGFR,5297,dacomitinib,True,1,8.222,IC50,SCIENTIFIC LITERATURE,2,0,0,"Non-small cell lung cancer, positive for epide...",True,1.0
